# Live Signal Generation

This notebook demonstrates:
- Connecting to kdb+ Tickerplant (TP) via PyKX
- Subscribing to real-time trade stream
- Generating live strategy signals using `BaseStrategy`
- Visualizing signals in real-time
- Discussion of the live → backtest → live validation loop

**Note**: This notebook can run with mock data if kdb+ is not available.

In [ ]:
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    sys.path.insert(0, str(Path.cwd().parent))

import time
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from medusa.strategies.examples.sma_crossover import SMACrossoverStrategy

try:
    import pykx as kx
    HAS_PYKX = True
except ImportError:
    HAS_PYKX = False
    print("PyKX not installed. Using mock data mode.")
    print("Install PyKX with: pip install pykx")

## 1. Connect to kdb+ Tickerplant (or use mock data)

In [ ]:
USE_MOCK = not HAS_PYKX  # Use mock data if PyKX not available

if HAS_PYKX and not USE_MOCK:
    try:
        # Connect to RDB (port 5011) for recent data
        conn = kx.SyncQConnection(host="localhost", port=5011, timeout=5.0)
        print("✅ Connected to kdb+ RDB at localhost:5011")
        USE_MOCK = False
    except Exception as e:
        print(f"⚠️ Could not connect to kdb+: {e}")
        print("Falling back to mock data mode")
        USE_MOCK = True
else:
    print("Using mock data mode")

## 2. Fetch Real-Time Trade Data

In [ ]:
def fetch_live_data_kdb(conn, symbol="BTCUSD", lookback_minutes=60):
    """Fetch recent trade data from kdb+ RDB.
    
    Args:
        conn: PyKX connection to RDB
        symbol: Trading symbol
        lookback_minutes: How far back to fetch
        
    Returns:
        DataFrame with OHLCV data
    """
    # Query: aggregate trades into 1-minute OHLCV bars
    query = f"""
    select 
        open: first price,
        high: max price,
        low: min price,
        close: last price,
        volume: sum size
    by 1 xbar timestamp.minute from trade
    where sym=`{symbol}, timestamp > .z.p - 00:00:{lookback_minutes}:00.000
    """

    result = conn(query)
    df = result.pd()
    df = df.set_index('timestamp')

    return df

def generate_mock_live_data(n_bars=60):
    """Generate mock live data for demo purposes."""
    np.random.seed(int(time.time()))

    # Start from current time
    end_time = datetime.now()
    start_time = end_time - timedelta(minutes=n_bars)
    dates = pd.date_range(start_time, end_time, freq="1min")

    # Trending price with noise
    t = np.arange(n_bars)
    trend = 0.5 * t
    noise = np.random.normal(0, 5, n_bars)
    price = 50000 + trend + noise

    # OHLC
    high = price * (1 + np.abs(np.random.normal(0, 0.002, n_bars)))
    low = price * (1 - np.abs(np.random.normal(0, 0.002, n_bars)))
    open_price = price + np.random.normal(0, 2, n_bars)
    volume = np.random.randint(10, 100, n_bars)

    df = pd.DataFrame(
        {
            "open": open_price,
            "high": high,
            "low": low,
            "close": price,
            "volume": volume,
        },
        index=dates[:n_bars],
    )

    return df

# Fetch data
if USE_MOCK:
    df = generate_mock_live_data(n_bars=100)
    print(f"Generated {len(df)} bars of mock data")
else:
    df = fetch_live_data_kdb(conn, symbol="BTCUSD", lookback_minutes=100)
    print(f"Fetched {len(df)} bars from kdb+ RDB")

df.tail()

## 3. Initialize Strategy and Generate Live Signals

In [ ]:
# Initialize strategy
strategy = SMACrossoverStrategy(fast_period=10, slow_period=30)

# Generate signals on full buffer
entries, exits = strategy.generate_signals(df)

print("\nSignals generated:")
print(f"  Entries: {entries.sum()}")
print(f"  Exits: {exits.sum()}")

# Get most recent signal
latest_signal = strategy.generate_signal(df)

if latest_signal:
    print(f"\n🔔 LIVE SIGNAL: {latest_signal['action']} at {latest_signal['price']:.2f}")
else:
    print("\n⏸️  No signal at current bar")

## 4. Visualize Live Signals with Plotly

In [ ]:
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("Plotly not installed. Using matplotlib instead.")

In [ ]:
if HAS_PLOTLY:
    # Calculate SMAs for visualization
    from medusa.features.technical import sma
    fast_sma = sma(df["close"], 10)
    slow_sma = sma(df["close"], 30)

    # Create interactive plot
    fig = go.Figure()

    # Price candlestick
    fig.add_trace(go.Candlestick(
        x=df.index,
        open=df['open'],
        high=df['high'],
        low=df['low'],
        close=df['close'],
        name='Price'
    ))

    # SMAs
    fig.add_trace(go.Scatter(
        x=df.index, y=fast_sma,
        mode='lines',
        name='Fast SMA (10)',
        line=dict(color='blue', width=1.5)
    ))

    fig.add_trace(go.Scatter(
        x=df.index, y=slow_sma,
        mode='lines',
        name='Slow SMA (30)',
        line=dict(color='orange', width=1.5)
    ))

    # Entry signals
    entry_idx = df.index[entries]
    entry_prices = df.loc[entries, 'close']
    fig.add_trace(go.Scatter(
        x=entry_idx, y=entry_prices,
        mode='markers',
        name='Buy',
        marker=dict(symbol='triangle-up', size=12, color='green')
    ))

    # Exit signals
    exit_idx = df.index[exits]
    exit_prices = df.loc[exits, 'close']
    fig.add_trace(go.Scatter(
        x=exit_idx, y=exit_prices,
        mode='markers',
        name='Sell',
        marker=dict(symbol='triangle-down', size=12, color='red')
    ))

    # Layout
    fig.update_layout(
        title='Live Trading Signals (SMA Crossover)',
        xaxis_title='Time',
        yaxis_title='Price',
        height=600,
        hovermode='x unified',
        xaxis_rangeslider_visible=False
    )

    fig.show()
else:
    # Fallback to matplotlib
    from medusa.features.technical import sma
    fast_sma = sma(df["close"], 10)
    slow_sma = sma(df["close"], 30)

    plt.figure(figsize=(14, 7))
    plt.plot(df.index, df["close"], label="Close Price", alpha=0.7)
    plt.plot(df.index, fast_sma, label="Fast SMA (10)", linewidth=1.5)
    plt.plot(df.index, slow_sma, label="Slow SMA (30)", linewidth=1.5)

    plt.scatter(df.index[entries], df.loc[entries, "close"],
                color="green", marker="^", s=100, label="Buy", zorder=5)
    plt.scatter(df.index[exits], df.loc[exits, "close"],
                color="red", marker="v", s=100, label="Sell", zorder=5)

    plt.title("Live Trading Signals (SMA Crossover)")
    plt.xlabel("Time")
    plt.ylabel("Price")
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 5. Simulate Real-Time Signal Stream

Show how signals would be generated bar-by-bar in production.

In [ ]:
print("Simulating real-time signal generation...\n")
print("Time                      | Price    | Signal")
print("-" * 55)

# Replay last 20 bars
for i in range(len(df) - 20, len(df)):
    # Sliding window: all data up to current bar
    current_data = df.iloc[:i+1]

    # Generate signal
    signal = strategy.generate_signal(current_data)

    timestamp = current_data.index[-1]
    price = current_data["close"].iloc[-1]

    if signal:
        action = signal["action"].upper()
        emoji = "🟢" if action == "BUY" else "🔴"
        print(f"{timestamp} | {price:8.2f} | {emoji} {action}")
    else:
        print(f"{timestamp} | {price:8.2f} | ⏸️  HOLD")

print("-" * 55)

## 6. The Live → Backtest → Live Validation Loop

### Why this loop is critical:

1. **Backtest** (historical data):
   - Develop strategy on past data
   - Optimize parameters
   - Measure risk/return metrics
   - **Problem**: Risk of overfitting to specific market regime

2. **Paper Trading** (live data, no real money):
   - Test strategy on real-time data stream
   - Verify signal generation logic
   - Check for data issues (gaps, latency, bad ticks)
   - **Problem**: Slippage and execution uncertainty still unknown

3. **Live Trading** (real money, small size):
   - Execute strategy with real capital
   - Measure actual slippage, fees, and execution quality
   - Monitor for regime changes
   - **Problem**: Market impact and psychological factors

4. **Feedback Loop**:
   - Collect live execution data
   - Feed back into backtester to refine assumptions
   - Adjust slippage/fee models
   - Re-optimize if performance degrades

### Key differences between backtest and live:

| Aspect | Backtest | Live |
|--------|----------|------|
| Data quality | Clean, complete | Gaps, bad ticks, latency |
| Execution | Instant, mid-price | Slippage, queue priority |
| Slippage | Modeled (constant %) | Variable, order-dependent |
| Latency | Zero | Network + exchange delay |
| Psychology | None | Fear, greed, hesitation |
| Market impact | Assumed small | Can be large for illiquid assets |

### Medusa workflow:

```
HDB (historical) ──► Backtest (VectorBT) ──► Optimize (Optuna)
     │                                            │
     │                                            ▼
     │                                    Best parameters
     │                                            │
     ▼                                            ▼
RDB (recent) ───► Paper trade ────► If good ───► Live trade (small size)
     │                                            │
     │                                            ▼
     └────────────────────────── Execution data ─┘
                                        │
                                        ▼
                                Refine assumptions
```

## 7. Next Steps

- Implement order publisher to send signals to kdb+ strategy engine
- Add position sizing logic (Kelly criterion, fixed fractional)
- Build risk monitoring dashboard (VAR, exposure limits)
- Set up alerting (Discord/Telegram) for critical signals
- Collect execution data for slippage analysis

In [ ]:
# Close kdb+ connection if open
if not USE_MOCK and HAS_PYKX:
    conn.close()
    print("✅ Closed kdb+ connection")